# Rag Arena Experiments

This notebook demonstrates the multi-dataset RAG workflow used by Rag Arena.

In [1]:
import json
import os
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent

api_keys_path = project_root / 'api_keys.json'
if api_keys_path.exists():
    api_keys = json.loads(api_keys_path.read_text(encoding='utf-8'))
    if api_keys.get('OPENROUTER_API_KEY'):
        os.environ['OPENROUTER_API_KEY'] = api_keys['OPENROUTER_API_KEY']

from rag_arena.data import load_qa_split
from rag_arena.experiments import ExperimentConfig, run_experiment
from rag_arena.indexing import build_corpus
from rag_arena.retrieval import build_retriever

/Volumes/HowesT7/NTU-Course-Materials/Assignments/AI6130 - Large Language Models/AI6130-GroupProject/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
samples = load_qa_split(dataset_name='hotpotqa', dataset_config='distractor', split='validation', sample_size=5, seed=42)
documents = build_corpus(samples)
retriever = build_retriever(documents, {'method': 'bm25', 'top_k': 5})
samples[0].question, retriever.retrieve(samples[0].question, top_k=3)[0].metadata

('What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?',
 {'dataset_name': 'hotpotqa',
  'sample_id': '5a7a567255429941d65f25bd',
  'title': 'Iqbal F. Qadir',
  'is_supporting_doc': True,
  'doc_id': '5a7a567255429941d65f25bd:Iqbal F. Qadir',
  'sentences': ['Vice-Admiral Iqbal Fazl Quadir (Urdu:اقبال فضل قادر) , is a retired three-star rank admiral in the Pakistan Navy, former diplomat, and a defence analyst.',
   'He is renown for his participation in second war with India when he was part of the flotilla that attacked the radar station in Dwarka, India.']})

In [3]:
hotpot_config = ExperimentConfig(
    dataset_name='hotpotqa',
    dataset_config='distractor',
    sample_size=5,
    output_dir='outputs/notebook_hotpot_smoke',
    retriever_config={
        'method': 'hybrid',
        'top_k': 2,
        'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
        'alpha': 0.5,
    },
    rerank_config={
        'enabled': False,
    },
    generation_config={
        'provider': 'ollama',
        'model_name': 'qwen3.5:2b',
        'temperature': 0.0,
        'max_tokens': 8192,
    },
)
hotpot_result = run_experiment(hotpot_config)
hotpot_result.summary

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2961.27it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⏳Initializing ollama model...
✅ollama model loaded！


{'dataset_name': 'hotpotqa',
 'num_samples': 5,
 'exact_match': 0.2,
 'answer_f1': 0.2,
 'retrieval_recall_at_k': 0.8,
 'supporting_title_f1': 0.8,
 'supporting_sentence_f1': 0.3466666666666667}

In [9]:
wiki2_config = ExperimentConfig(
    dataset_name='2wikimultihopqa',
    dataset_split='validation',
    sample_size=5,
    output_dir='outputs/notebook_2wiki_smoke',
    retriever_config={
        'method': 'iterative',
        'base_method': 'hybrid',
        'top_k': 10,
        'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
        'alpha': 0.5,
        'max_iter': 3,
        'sim_threshold': 0.85,
    },
    rerank_config={
        'enabled': True,
        'model_name': 'cross-encoder/ms-marco-MiniLM-L-6-v2',
        'top_k': 5,
    },
    generation_config={
        'provider': 'openrouter',
        'model_name': 'qwen/qwen3.6-plus-preview:free',
        'temperature': 0.0,
        'max_tokens': 8192,
    },
)
wiki2_result = run_experiment(wiki2_config)
wiki2_result.summary

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 607.37it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13242.19it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 10270.09it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-------------------------

⏳Initializing openrouter model...
✅openrouter model loaded！


{'dataset_name': '2wikimultihopqa',
 'num_samples': 5,
 'exact_match': 0.8,
 'answer_f1': 0.8,
 'retrieval_recall_at_k': 0.85,
 'supporting_title_f1': 0.5333333333333334,
 'supporting_sentence_f1': 0.20331797789692527}